## **Modelo gpt-4.1 para los primeros 7 turnos**

In [ ]:
from openai import OpenAI
import pandas as pd

# Cargar dataset
df = pd.read_csv('conversations_dataset_readable.csv')

# Seleccionar solo los primeros 7 turnos
df_first_7 = df.head(7).copy()

# Inicializar cliente OpenAI
client = OpenAI(api_key="")

def generate_response(row):
    """Genera respuesta usando OpenAI con tu prompt personalizado"""

    prompt = f"""
You are a knowledgeable and helpful AI assistant. Your task is to provide accurate, helpful responses to user questions based on the available information.

CONTEXT PROCESSING:
- If you do not know the answer, respond only with: "I'm sorry, but I don't have the answer to your question."
- Do not give any extra explanations about why you don't know or where to obtain the information.
- If context IS available (contexts_available = True), carefully analyze the provided contexts to formulate your response

CONVERSATION HANDLING:
- Maintain conversation history awareness across multiple turns
- For Follow-up questions: Connect to previous discussion while providing complete answers
- For Clarification questions: Address any ambiguities from previous turns with clear explanations
- For N/A (first turn): Provide comprehensive initial responses

RESPONSE STYLE BY QUESTION TYPE:
- Factoid: Provide direct, concise factual answers with key details
- Explanation: Offer detailed, educational explanations with context and examples
- Summarization: Synthesize information into clear, organized summaries
- How-To: Provide step-by-step instructions with practical guidance
- Comparative: Present balanced comparisons with clear differentiators
- Opinion: Offer reasoned perspectives while acknowledging subjectivity
- Composite: Address multiple aspects comprehensively
- Keyword: Expand on key concepts with relevant information
- Non-Question: Interpret the intent and provide helpful information
- Troubleshooting: Diagnose issues and provide practical solutions

ANSWERABILITY HANDLING:
- ANSWERABLE: Provide complete, well-supported answers using available contexts
- PARTIAL: Acknowledge limitations while providing the available information
- UNANSWERABLE: Clearly state the inability to answer when no context exists
- CONVERSATIONAL: Engage naturally while maintaining helpfulness

RESPONSE QUALITY STANDARDS:
- Be accurate, factual, and evidence-based
- Maintain clear, professional, and helpful tone
- Structure responses for easy comprehension
- Acknowledge uncertainties when appropriate
- Connect multi-turn conversations coherently

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Based on the above, provide your best response:
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=256
    )

    response = completion.choices[0].message.content.strip()
    return response



#   EJECUCIÓN PRINCIPAL

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    print(f"Procesando turno {idx + 1}...")

    try:
        generated = generate_response(row)

        print(f"\n=== TURNO {row['turn']} ===")
        print(f"PREGUNTA: {row['current_question']}")
        print(f"RESPUESTA GENERADA: {generated}")
        print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
        print("-" * 80)

    except Exception as e:
        print(f"Error en turno {idx + 1}: {e}")


print("\n" + "="*80)
print("RESUMEN EJECUCIÓN COMPLETADA")
print("="*80)

Generando respuestas para los primeros 7 turnos...

Procesando turno 1...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
Procesando turno 2...

=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: Yes, during the 2017 season, the Arizona Cardinals played a game outside the US. They faced the Los Angeles Rams at Twickenham Stadium in London, England as part of the NFL International Series. This was the Cardinals' first appearance in the International Series.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
------------------

**Evaluación por turno:**

In [ ]:
import evaluate
import numpy as np
from scipy.stats import hmean


# RESPUESTAS GENERADAS Y OBJETIVO
generated_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "Yes, during the 2017 season, the Arizona Cardinals played a game outside the US. They faced the Los Angeles Rams at Twickenham Stadium in London, England as part of the NFL International Series. This was the Cardinals' first appearance in the International Series.",
    "Yes, the Arizona Cardinals and the Chicago Cardinals are the same franchise. The team was originally founded as the Chicago Cardinals and played in Chicago until 1959. In 1960, the team moved to St. Louis and became the St. Louis Cardinals. Later, in 1988, the franchise relocated to Arizona and is now known as the Arizona Cardinals. So, while the name and location have changed over time, it is the same continuous NFL team.",
    "There are 32 teams in the NFL. The league is divided into two conferences: the American Football Conference (AFC) and the National Football Conference (NFC), each with 16 teams.",
    "There are 12 teams in the NFL playoffs: six from the American Football Conference (AFC) and six from the National Football Conference (NFC).",
    "The Pittsburgh Steelers have won the most Super Bowls, with a total of six championships. The Dallas Cowboys, New England Patriots, and San Francisco 49ers each have five Super Bowl victories.",
    "The New England Patriots have played in the Super Bowl 10 times."
]

reference_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.",
    "The Chicago Cardinals became the St. Louis Cardinals in 1960 and eventually moved and became the Arizona Cardinals. The Chicago Cardinals ( now the Arizona Cardinals ) were a founding member of the NFL.",
    "There are 32 teams in the National Football League (NFL).",
    "Six teams from each conference (AFC and NFC), for a total of 12 team playoff system.",
    "The Pittsburgh Steelers have won six Super Bowls.",
    "The New England Patriots have the most Super Bowl appearances. Super Bowl LII was the Patriots ' tenth Super Bowl appearance."
]

# FUNCION RB_alg POR TURNO
def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))  # duplicar para cada turno

    # Media armónica por turno
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# CALCULO RB_alg POR TURNO
rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

# Mostrar resultados
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

Turno 1: RB_alg = 0.7258
Turno 2: RB_alg = 0.6016
Turno 3: RB_alg = 0.5671
Turno 4: RB_alg = 0.6047
Turno 5: RB_alg = 0.5533
Turno 6: RB_alg = 0.6128
Turno 7: RB_alg = 0.6158


**Media de todos los turnos**

In [ ]:
import evaluate
import numpy as np
from scipy.stats import hmean

#   RESPUESTAS GENERADAS
generated_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "Yes, during the 2017 season, the Arizona Cardinals played a game outside the US. They faced the Los Angeles Rams at Twickenham Stadium in London, England as part of the NFL International Series. This was the Cardinals' first appearance in the International Series.",
    "Yes, the Arizona Cardinals and the Chicago Cardinals are the same franchise. The team was originally founded as the Chicago Cardinals and played in Chicago until 1959. In 1960, the team moved to St. Louis and became the St. Louis Cardinals. Later, in 1988, the franchise relocated to Arizona and is now known as the Arizona Cardinals. So, while the name and location have changed over time, it is the same continuous NFL team.",
    "There are 32 teams in the NFL. The league is divided into two conferences: the American Football Conference (AFC) and the National Football Conference (NFC), each with 16 teams.",
    "There are 12 teams in the NFL playoffs: six from the American Football Conference (AFC) and six from the National Football Conference (NFC).",
    "The Pittsburgh Steelers have won the most Super Bowls, with a total of six championships. The Dallas Cowboys, New England Patriots, and San Francisco 49ers each have five Super Bowl victories.",
    "The New England Patriots have played in the Super Bowl 10 times."
]

#   RESPUESTAS OBJETIVO
reference_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.",
    "The Chicago Cardinals became the St. Louis Cardinals in 1960 and eventually moved and became the Arizona Cardinals. The Chicago Cardinals ( now the Arizona Cardinals ) were a founding member of the NFL.",
    "There are 32 teams in the National Football League (NFL).",
    "Six teams from each conference (AFC and NFC), for a total of 12 team playoff system.",
    "The Pittsburgh Steelers have won six Super Bowls.",
    "The New England Patriots have the most Super Bowl appearances. Super Bowl LII was the Patriots ' tenth Super Bowl appearance."
]


#   FUNCION RB_alg
def compute_rb_alg(preds, refs):
    """
    Calcula RB_alg como media armónica de:
    - BERTScore F1
    - ROUGE-L F1
    - Precision BERT-K (aproximada como BERTScore precision)
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    # Acceder correctamente a ROUGE-L F1
    rouge_l_f1 = np.array([rouge_results["rougeL"]])
    if rouge_l_f1.shape[0] == 1:
        # Expandir para que tenga mismo tamaño que bert_f1
        rouge_l_f1 = np.full_like(bert_f1, rouge_l_f1[0])

    # Media armónica RB_alg
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)  # evitar ceros exactos
    rb_alg_scores = hmean(combined, axis=0)

    return np.mean(rb_alg_scores)

#   CALCULO RB_alg
rb_alg_score = compute_rb_alg(generated_responses, reference_responses)
print(f"RB_alg score: {rb_alg_score:.4f}")


RB_alg score: 0.6116


## **Modelo command-a-03-2025 de Cohere con api de la profesora**

In [ ]:
import cohere
import pandas as pd
import numpy as np
from scipy.stats import hmean
import evaluate

# ==========================
#   CARGA DE DATASET
# ==========================

df = pd.read_csv("conversations_dataset_readable.csv")
df_first_7 = df.head(7).copy()

# ==========================
#   CLIENTE COHERE
# ==========================

co = cohere.Client("COHERE API")


# ==========================
#   PROMPT IDEAL DEFINITIVO
# ==========================

def generate_response(row):

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Give the final answer in the exact style described above.
"""

    # ==========================
    #   LLAMADA A COHERE
    # ==========================

    response = co.chat(
        model="command-a-03-2025",
        message=prompt,
        temperature=0.0,
        max_tokens=150
    )

    return response.text.strip()


# ==========================
#   GENERACIÓN DE RESPUESTAS
# ==========================

generated_responses = []
reference_responses = df_first_7["target_answer"].tolist()

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    gen = generate_response(row)
    generated_responses.append(gen)

    print(f"=== TURNO {row['turn']} ===")
    print(f"PREGUNTA: {row['current_question']}")
    print(f"RESPUESTA GENERADA: {gen}")
    print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
    print("-" * 80)


# ==============================================================================
#   FUNCIÓN DE MÉTRICA RB_alg POR TURNO (BERTScore + ROUGE-L + BERT Precision)
# ==============================================================================

def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))

    # Media armónica
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# ==============================
#   CÁLCULO FINAL DE MÉTRICAS
# ==============================

rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

print("\nRB_alg POR TURNO")
print("=" * 40)
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

media_rb_alg = np.mean(rb_alg_scores_per_turn)
print("\n====================================")
print(f"MEDIA GLOBAL RB_alg = {media_rb_alg:.4f}")
print("====================================")

Generando respuestas para los primeros 7 turnos...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: The Arizona Cardinals played one game outside the US in 2017, against the Los Angeles Rams at Twickenham Stadium in London, England, as part of the NFL International Series. NFL = National Football League.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
=== TURNO 3 ===
PREGUNTA: Are the Arizona 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


RB_alg POR TURNO
Turno 1: RB_alg = 0.7669
Turno 2: RB_alg = 0.6403
Turno 3: RB_alg = 0.6075
Turno 4: RB_alg = 0.6410
Turno 5: RB_alg = 0.6070
Turno 6: RB_alg = 0.6309
Turno 7: RB_alg = 0.6024

MEDIA GLOBAL RB_alg = 0.6423


## **gpt-4 con un prompt diferente**

In [ ]:
from openai import OpenAI
import pandas as pd

# Cargar dataset
df = pd.read_csv('conversations_dataset_readable.csv')

# Seleccionar solo los primeros 7 turnos
df_first_7 = df.head(7).copy()

# Inicializar cliente OpenAI
client = OpenAI(api_key="")

def generate_response(row):
    """Genera respuesta usando OpenAI con tu prompt personalizado"""

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your primary objective is:
Produce responses that match the *style, length, structure, and level of detail* of the target answers.

STRICT INSTRUCTIONS:

1. **Style & Length**
   - Keep answers short and factual.
   - Do not add extra details that are not directly needed.
   - Avoid expansions, examples, comparisons, history, or additional context unless the question explicitly requires it.
   - Match the tone seen in the dataset: concise, neutral, direct.

2. **Use of Context**
   - If context is unavailable or insufficient, respond exactly with:
     “I'm sorry, but I don't have the answer to your question.”
   - Do NOT add explanations, caveats, or suggestions.

3. **Conversation Handling**
   - Use conversation history only to disambiguate follow-up questions.
   - Do NOT restate history.
   - Maintain short, direct responses even in multi-turn settings.

4. **Answerability**
   - ANSWERABLE: respond concisely using only necessary facts.
   - PARTIAL: give only what is known and remain concise.
   - UNANSWERABLE: use the required fallback text.

5. **No Creativity**
   - Do not elaborate.
   - Do not add extra facts beyond what is essential.
   - Do not provide exhaustive explanations.

Now generate the answer following the above constraints:

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Based on the above, provide your best response:
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=256
    )

    response = completion.choices[0].message.content.strip()
    return response



#   EJECUCIÓN PRINCIPAL

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    print(f"Procesando turno {idx + 1}...")

    try:
        generated = generate_response(row)

        print(f"\n=== TURNO {row['turn']} ===")
        print(f"PREGUNTA: {row['current_question']}")
        print(f"RESPUESTA GENERADA: {generated}")
        print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
        print("-" * 80)

    except Exception as e:
        print(f"Error en turno {idx + 1}: {e}")


print("\n" + "="*80)
print("RESUMEN EJECUCIÓN COMPLETADA")
print("="*80)

Generando respuestas para los primeros 7 turnos...

Procesando turno 1...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
Procesando turno 2...

=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: Yes, the Arizona Cardinals played outside the US in London, England during the 2017 season.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
Procesando turno 3...

=== TURNO 3 ===
PREGUNTA: Are the Arizona Cardinals and the Chicago Cardinals the same 

In [ ]:
#pip install bert_score

In [ ]:
#pip install evaluate rouge-score


### **Evaluación por turno**

In [ ]:
import evaluate
import numpy as np
from scipy.stats import hmean


# RESPUESTAS GENERADAS Y OBJETIVO
generated_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "Yes, the Arizona Cardinals played outside the US in London, England during the 2017 season.",
    "Yes, the Arizona Cardinals and the Chicago Cardinals are the same team.",
    "There are 32 teams in the NFL.",
    "There are 12 teams in the NFL playoffs.",
    "The Pittsburgh Steelers have won the most Super Bowls, with six titles.",
    "The New England Patriots have played in ten Super Bowls."
]

reference_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.",
    "The Chicago Cardinals became the St. Louis Cardinals in 1960 and eventually moved and became the Arizona Cardinals. The Chicago Cardinals ( now the Arizona Cardinals ) were a founding member of the NFL.",
    "There are 32 teams in the National Football League (NFL).",
    "Six teams from each conference (AFC and NFC), for a total of 12 team playoff system.",
    "The Pittsburgh Steelers have won six Super Bowls.",
    "The New England Patriots have the most Super Bowl appearances. Super Bowl LII was the Patriots ' tenth Super Bowl appearance."
]

# FUNCION RB_alg POR TURNO
def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))  # duplicar para cada turno

    # Media armónica por turno
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# CALCULO RB_alg POR TURNO
rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

# Mostrar resultados
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Turno 1: RB_alg = 0.7734
Turno 2: RB_alg = 0.6498
Turno 3: RB_alg = 0.5787
Turno 4: RB_alg = 0.7368
Turno 5: RB_alg = 0.5642
Turno 6: RB_alg = 0.7009
Turno 7: RB_alg = 0.6578


### **Media de los 7 turnos**

In [ ]:
import evaluate
import numpy as np
from scipy.stats import hmean

#   RESPUESTAS GENERADAS
generated_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "Yes, the Arizona Cardinals played outside the US in London, England during the 2017 season.",
    "Yes, the Arizona Cardinals and the Chicago Cardinals are the same team.",
    "There are 32 teams in the NFL.",
    "There are 12 teams in the NFL playoffs.",
    "The Pittsburgh Steelers have won the most Super Bowls, with six titles.",
    "The New England Patriots have played in ten Super Bowls."
]

#   RESPUESTAS OBJETIVO
reference_responses = [
    "I'm sorry, but I don't have the answer to your question.",
    "The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.",
    "The Chicago Cardinals became the St. Louis Cardinals in 1960 and eventually moved and became the Arizona Cardinals. The Chicago Cardinals ( now the Arizona Cardinals ) were a founding member of the NFL.",
    "There are 32 teams in the National Football League (NFL).",
    "Six teams from each conference (AFC and NFC), for a total of 12 team playoff system.",
    "The Pittsburgh Steelers have won six Super Bowls.",
    "The New England Patriots have the most Super Bowl appearances. Super Bowl LII was the Patriots ' tenth Super Bowl appearance."
]


#   FUNCION RB_alg
def compute_rb_alg(preds, refs):
    """
    Calcula RB_alg como media armónica de:
    - BERTScore F1
    - ROUGE-L F1
    - Precision BERT-K (aproximada como BERTScore precision)
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    # Acceder correctamente a ROUGE-L F1
    rouge_l_f1 = np.array([rouge_results["rougeL"]])
    if rouge_l_f1.shape[0] == 1:
        # Expandir para que tenga mismo tamaño que bert_f1
        rouge_l_f1 = np.full_like(bert_f1, rouge_l_f1[0])

    # Media armónica RB_alg
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)  # evitar ceros exactos
    rb_alg_scores = hmean(combined, axis=0)

    return np.mean(rb_alg_scores)

#   CALCULO RB_alg
rb_alg_score = compute_rb_alg(generated_responses, reference_responses)
print(f"RB_alg score: {rb_alg_score:.4f}")

RB_alg score: 0.6659


## **gpt-4 con 80 turnos**

In [ ]:
from openai import OpenAI
import pandas as pd

# Cargar dataset
df = pd.read_csv('conversations_dataset_readable.csv')

# Seleccionar solo los primeros 7 turnos
df_first_7 = df.head(80).copy()

# Inicializar cliente OpenAI
client = OpenAI(api_key="")

def generate_response(row):
    """Genera respuesta usando OpenAI con tu prompt personalizado"""

    prompt = f"""
You are an AI assistant participating in a shared task to generate responses consistent with the target answer style found in the dataset.

The dataset follows strict annotation rules. You must adapt your response depending on the metadata provided.

=====================
STYLE GUIDELINES
=====================
• Responses must be short, factual, and direct.
• Do NOT add any information that is not required by the question.
• When context provides multiple facts, include only the ones directly relevant.
• When answering factoid questions, include numbers, years, places, or names if present in context.
• Prefer simple declarative sentences.
• Avoid explanations, reasoning steps, and historical background unless in context.

=====================
ANSWERABILITY RULES
=====================
If answerability = "UNANSWERABLE":
   → Respond exactly with:
   "I'm sorry, but I don't have the answer to your question."

If answerability = "PARTIAL":
   → Provide only the facts that are explicitly supported.

If answerability = "ANSWERABLE":
   → Answer concisely using only the necessary facts.

=====================
QUESTION TYPE RULES
=====================
If question_type = "Factoid":
    - Provide a direct, concise fact.
If question_type = "Explanation":
    - Provide a short explanation (1–2 sentences).
If question_type = "How-To":
    - Give a very short step-style answer.
If question_type = "Comparative":
    - Provide a short comparison using only context facts.
If question_type = "Summarization":
    - Output a 1–2 sentence summary.
If question_type = "Keyword":
    - Provide a short definition or clarification.
If question_type = "Composite":
    - Address all required parts briefly.
If question_type = "Non-Question":
    - Interpret intent and respond minimally.
If question_type = "Troubleshooting":
    - Provide a short corrective suggestion.

=====================
MULTI-TURN LOGIC
=====================
• If multi_turn = "Follow-up":
      Use prior turns ONLY to disambiguate the current question.
• If multi_turn = "Clarification":
      Clarify missing detail concisely.
• If multi_turn = "Conversational":
      Respond normally but stay concise.
• If multi_turn = "N/A":
      Treat the question as standalone.

Never restate conversation history.

=====================
CONTEXT HANDLING
=====================
If contexts_text is empty or irrelevant:
    Respond with the UNANSWERABLE fallback.

If contexts_text contains multiple facts:
    Use only the ones needed for the question.

Now generate the answer following the above constraints:

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Based on the above, provide your best response:
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "user", "content": prompt}
        ],
        #tenia 0.7 y le puse 0.8
        temperature=0.8,
        max_tokens=256
    )

    response = completion.choices[0].message.content.strip()
    return response



#   EJECUCIÓN PRINCIPAL

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    print(f"Procesando turno {idx + 1}...")

    try:
        generated = generate_response(row)

        print(f"\n=== TURNO {row['turn']} ===")
        print(f"PREGUNTA: {row['current_question']}")
        print(f"RESPUESTA GENERADA: {generated}")
        print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
        print("-" * 80)

    except Exception as e:
        print(f"Error en turno {idx + 1}: {e}")


print("\n" + "="*80)
print("RESUMEN EJECUCIÓN COMPLETADA")
print("="*80)

Generando respuestas para los primeros 7 turnos...

Procesando turno 1...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
Procesando turno 2...

=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: The Arizona Cardinals played outside the US at Twickenham Stadium in London, England, during the 2017 season.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
Procesando turno 3...

=== TURNO 3 ===
PREGUNTA: Are the Arizona Cardinals and the Chicago C

## **gpt-4 con 20 y 7 turnos ya con pipeline de prueba y otros prompts**

In [ ]:
from openai import OpenAI
import pandas as pd
import numpy as np
from scipy.stats import hmean
import evaluate

# ==========================
#   CARGA DE DATASET
# ==========================

df = pd.read_csv("conversations_dataset_readable.csv")
df_first_7 = df.head(20).copy()

# ==========================
#   CLIENTE OPENAI
# ==========================

client = OpenAI(api_key="")


# ==========================
#   PROMPT IDEAL DEFINITIVO
# ==========================

def generate_response(row):

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Give the final answer in the exact style described above.
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )

    return completion.choices[0].message.content.strip()


# ==========================
#   GENERACIÓN DE RESPUESTAS
# ==========================

generated_responses = []
reference_responses = df_first_7["target_answer"].tolist()

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    gen = generate_response(row)
    generated_responses.append(gen)

    print(f"=== TURNO {row['turn']} ===")
    print(f"PREGUNTA: {row['current_question']}")
    print(f"RESPUESTA GENERADA: {gen}")
    print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
    print("-" * 80)


# ==============================================================================
#   FUNCIÓN DE MÉTRICA RB_alg POR TURNO (BERTScore + ROUGE-L + BERT Precision)
# ==============================================================================

def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))

    # Media armónica
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# ==============================
#   CÁLCULO FINAL DE MÉTRICAS
# ==============================

rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

print("\nRB_alg POR TURNO")
print("=" * 40)
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

media_rb_alg = np.mean(rb_alg_scores_per_turn)
print("\n====================================")
print(f"MEDIA GLOBAL RB_alg = {media_rb_alg:.4f}")
print("====================================")


Generando respuestas para los primeros 7 turnos...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: The Arizona Cardinals played outside the US in 2017 when they faced the Los Angeles Rams at Twickenham Stadium in London, England, as part of the NFL International Series.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
=== TURNO 3 ===
PREGUNTA: Are the Arizona Cardinals and the Chicago Cardinal

In [ ]:
from openai import OpenAI
import pandas as pd
import numpy as np
from scipy.stats import hmean
import evaluate

# ==========================
#   CARGA DE DATASET
# ==========================

df = pd.read_csv("conversations_dataset_readable.csv")
df_first_7 = df.head(7).copy()

# ==========================
#   CLIENTE OPENAI
# ==========================

client = OpenAI(api_key="")


# ==========================
#   PROMPT IDEAL DEFINITIVO
# ==========================

def generate_response(row):

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Give the final answer in the exact style described above.
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )

    return completion.choices[0].message.content.strip()


# ==========================
#   GENERACIÓN DE RESPUESTAS
# ==========================

generated_responses = []
reference_responses = df_first_7["target_answer"].tolist()

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    gen = generate_response(row)
    generated_responses.append(gen)

    print(f"=== TURNO {row['turn']} ===")
    print(f"PREGUNTA: {row['current_question']}")
    print(f"RESPUESTA GENERADA: {gen}")
    print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
    print("-" * 80)


# ==============================================================================
#   FUNCIÓN DE MÉTRICA RB_alg POR TURNO (BERTScore + ROUGE-L + BERT Precision)
# ==============================================================================

def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))

    # Media armónica
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# ==============================
#   CÁLCULO FINAL DE MÉTRICAS
# ==============================

rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

print("\nRB_alg POR TURNO")
print("=" * 40)
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

media_rb_alg = np.mean(rb_alg_scores_per_turn)
print("\n====================================")
print(f"MEDIA GLOBAL RB_alg = {media_rb_alg:.4f}")
print("====================================")

Generando respuestas para los primeros 7 turnos...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: The Arizona Cardinals played outside the US in 2017 when they faced the Los Angeles Rams at Twickenham Stadium in London, England, as part of the NFL International Series.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
=== TURNO 3 ===
PREGUNTA: Are the Arizona Cardinals and the Chicago Cardinal

In [ ]:
from openai import OpenAI
import pandas as pd
import numpy as np
from scipy.stats import hmean
import evaluate

# ==========================
#   CARGA DE DATASET
# ==========================

df = pd.read_csv("conversations_dataset_readable.csv")
df_first_7 = df.head(20).copy()

# ==========================
#   CLIENTE OPENAI
# ==========================

client = OpenAI(api_key="")


# ==========================
#   PROMPT IDEAL DEFINITIVO
# ==========================

def generate_response(row):

    prompt = f"""
You are an AI system participating in a supervised evaluation task.

Your goal is to generate an answer that **matches as closely as possible** the writing style, length, structure, tone, directness, level of detail, and informational density of the reference answer.

You will receive:
- current_question: the user’s question
- question_type: (Factoid, Explanation, Definition, Procedure, etc.)
- multi_turn: (Follow-up, Clarification, etc.) or empty
- answerability: (ANSWERABLE, UNANSWERABLE)
- reference_answer: the target answer you must stylistically imitate

STRICT GENERATION RULES:
1. **Your answer must resemble the reference_answer in structure and style**, not in content.
2. Match:
   - sentence length
   - paragraph structure
   - level of explanation
   - amount of detail
   - use of qualifiers
   - presence/absence of examples
   - overall brevity or verbosity
3. If answerability = UNANSWERABLE → respond with a short apology exactly like the reference answers typically do (“I'm sorry, but I don't have the answer to your question.”).
4. If question_type = Factoid → give short, direct facts with no elaboration unless the reference tends to elaborate.
5. If question_type = Explanation → produce 2–4 sentences with light detail, similar to references.
6. If multi_turn indicates Follow-up or Clarification → answer directly without repeating prior context unless reference answers usually repeat it (rare).
7. **Do not add information not implied by the reference answer’s style.**
8. **Never say “yes”, “sure”, or conversational fillers unless the reference systematically uses them (it does not).**
9. **Do not add extra facts or statistics unless the reference style typically does so.**
10. Maintain a neutral, factual tone with no enthusiasm or opinion.

Your output must ONLY be the generated answer — no explanations, no self-reference.

====================================================
INSTRUCTIONS
====================================================
Using ONLY the rules above, produce the final answer.

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Final Answer:

"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )

    return completion.choices[0].message.content.strip()


# ==========================
#   GENERACIÓN DE RESPUESTAS
# ==========================

generated_responses = []
reference_responses = df_first_7["target_answer"].tolist()

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    gen = generate_response(row)
    generated_responses.append(gen)

    print(f"=== TURNO {row['turn']} ===")
    print(f"PREGUNTA: {row['current_question']}")
    print(f"RESPUESTA GENERADA: {gen}")
    print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
    print("-" * 80)


# ==============================================================================
#   FUNCIÓN DE MÉTRICA RB_alg POR TURNO (BERTScore + ROUGE-L + BERT Precision)
# ==============================================================================

def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))

    # Media armónica
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# ==============================
#   CÁLCULO FINAL DE MÉTRICAS
# ==============================

rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

print("\nRB_alg POR TURNO")
print("=" * 40)
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

media_rb_alg = np.mean(rb_alg_scores_per_turn)
print("\n====================================")
print(f"MEDIA GLOBAL RB_alg = {media_rb_alg:.4f}")
print("====================================")

Generando respuestas para los primeros 7 turnos...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: The Arizona Cardinals primarily play their games in the United States, but in the 2017 season, they played one game outside the US. This game was part of the NFL International Series and took place at Twickenham Stadium in London, England.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
=== TURN

### **Intento de evaluar los 800 turnos (trono el programa)**

In [ ]:
from openai import OpenAI
import pandas as pd
import numpy as np
from scipy.stats import hmean
import evaluate

# ==========================
#   CARGA DE DATASET
# ==========================

df = pd.read_csv("conversations_dataset_readable.csv")
df_first_7 = df

# ==========================
#   CLIENTE OPENAI
# ==========================

client = OpenAI(api_key="")


# ==========================
#   PROMPT IDEAL DEFINITIVO
# ==========================

def generate_response(row):

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {row['current_question']}
Conversation History: {row['conversation_history_readable']}
Available Contexts: {row['contexts_text']}
Question Type: {row['question_type']}
Multi-turn Type: {row['multi_turn']}
Answerability: {row['answerability']}

Give the final answer in the exact style described above.
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )

    return completion.choices[0].message.content.strip()


# ==========================
#   GENERACIÓN DE RESPUESTAS
# ==========================

generated_responses = []
reference_responses = df_first_7["target_answer"].tolist()

print("Generando respuestas para los primeros 7 turnos...\n")

for idx, row in df_first_7.iterrows():
    gen = generate_response(row)
    generated_responses.append(gen)

    print(f"=== TURNO {row['turn']} ===")
    print(f"PREGUNTA: {row['current_question']}")
    print(f"RESPUESTA GENERADA: {gen}")
    print(f"RESPUESTA OBJETIVO: {row['target_answer']}")
    print("-" * 80)


# ==============================================================================
#   FUNCIÓN DE MÉTRICA RB_alg POR TURNO (BERTScore + ROUGE-L + BERT Precision)
# ==============================================================================

def compute_rb_alg_per_turn(preds, refs):
    """
    Calcula RB_alg individualmente para cada par predicción/referencia.
    """
    # BERTScore
    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=preds, references=refs, model_type="bert-base-uncased")
    bert_f1 = np.array(bert_results["f1"])
    bert_prec = np.array(bert_results["precision"])

    # ROUGE-L
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=preds, references=refs)
    rouge_l_f1 = np.array([rouge_results["rougeL"]] * len(preds))

    # Media armónica
    combined = np.vstack([bert_f1, rouge_l_f1, bert_prec])
    combined = np.clip(combined, 1e-8, 1.0)
    rb_alg_scores = hmean(combined, axis=0)

    return rb_alg_scores


# ==============================
#   CÁLCULO FINAL DE MÉTRICAS
# ==============================

rb_alg_scores_per_turn = compute_rb_alg_per_turn(generated_responses, reference_responses)

print("\nRB_alg POR TURNO")
print("=" * 40)
for i, score in enumerate(rb_alg_scores_per_turn, 1):
    print(f"Turno {i}: RB_alg = {score:.4f}")

media_rb_alg = np.mean(rb_alg_scores_per_turn)
print("\n====================================")
print(f"MEDIA GLOBAL RB_alg = {media_rb_alg:.4f}")
print("====================================")

Generando respuestas para los primeros 7 turnos...

=== TURNO 1 ===
PREGUNTA: where do the arizona cardinals play this week
RESPUESTA GENERADA: I'm sorry, but I don't have the answer to your question.
RESPUESTA OBJETIVO: I'm sorry, but I don't have the answer to your question.
--------------------------------------------------------------------------------
=== TURNO 2 ===
PREGUNTA: Do the Arizona Cardinals play outside the US?
RESPUESTA GENERADA: The Arizona Cardinals played outside the US in 2017 when they faced the Los Angeles Rams at Twickenham Stadium in London, England, as part of the NFL International Series.
RESPUESTA OBJETIVO: The Arizona Cardinals do play outside the United States. They had a game in London, England, on October 22, 2017, against the Los Angeles Rams at Twickenham Stadium and in 2005 they played in Mexico.
--------------------------------------------------------------------------------
=== TURNO 3 ===
PREGUNTA: Are the Arizona Cardinals and the Chicago Cardinal

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1 in organization org-GcoyUJ5YLeBHlzY3VRaTYoD6 on tokens per min (TPM): Limit 30000, Used 27743, Requested 2467. Please try again in 420ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
import json
import time
from openai import OpenAI
from openai import RateLimitError

# ==========================
#   CLIENTE OPENAI
# ==========================

client = OpenAI(api_key="")

# ==========================
#   PROMPT (SIN CAMBIOS)
# ==========================

def generate_response(task):

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {task['current_question']}
Conversation History: {task['conversation_history_readable']}
Available Contexts: {task['contexts_text']}
Question Type: {task['question_type']}
Multi-turn Type: {task['multi_turn']}
Answerability: {task['answerability']}

Give the final answer in the exact style described above.
"""

    while True:
        try:
            completion = client.chat.completions.create(
                model="gpt-5.1",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=150
            )
            return completion.choices[0].message.content.strip()

        except RateLimitError:
            # Backoff mínimo seguro
            time.sleep(0.6)

# ==========================
#   LECTURA Y ESCRITURA JSONL
# ==========================

input_path = "RAG.jsonl"
output_path = "responses-10.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for idx, line in enumerate(fin):
        task = json.loads(line)

        generated_text = generate_response(task)

        # Formato EXACTO requerido por la tarea
        task["predictions"] = [
            {
                "text": generated_text
            }
        ]

        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

        if (idx + 1) % 10 == 0:
            print(f"{idx + 1} turnos procesados")

print("✅ Archivo responses-10.jsonl generado correctamente.")


In [ ]:
# pip install openai


### Prueba con todos los turnos usando GPT-5.1

In [ ]:
import json
from openai import OpenAI

# ==========================
#   CLIENTE OPENAI
# ==========================
client = OpenAI(api_key="")

# ==========================
#   PREPROCESAMIENTO (MÍNIMO)
# ==========================
def get_current_question(task):
    # usualmente el último mensaje del input es la pregunta actual
    msgs = task.get("input", [])
    for m in reversed(msgs):
        if m.get("speaker") == "user" and m.get("text"):
            return m["text"]
    return ""

def format_conversation_history(task):
    msgs = task.get("input", [])
    lines = []
    for m in msgs:
        speaker = (m.get("speaker") or "unknown").upper()
        text = m.get("text") or ""
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)

def format_contexts(task):
    ctxs = task.get("contexts", [])
    parts = []
    for i, c in enumerate(ctxs, start=1):
        title = c.get("title") or ""
        text = c.get("text") or ""
        if title:
            parts.append(f"[Passage {i}] {title}\n{text}")
        else:
            parts.append(f"[Passage {i}] {text}")
    return "\n\n".join(parts)

def get_target_answer(task):
    t = task.get("targets", [])
    if isinstance(t, list) and len(t) > 0 and isinstance(t[0], dict):
        return t[0].get("text", "")
    return ""

# ==========================
#   PROMPT IDEAL DEFINITIVO (MISMO, SOLO CAMBIA DE DÓNDE SALE)
# ==========================
def generate_response(task):
    current_question = get_current_question(task)
    conversation_history = format_conversation_history(task)
    contexts_text = format_contexts(task)

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {current_question}
Conversation History: {conversation_history}
Available Contexts: {contexts_text}
Question Type: {task.get('Question Type')}
Multi-turn Type: {task.get('Multi-Turn')}
Answerability: {task.get('Answerability')}

Give the final answer in the exact style described above.
"""

    completion = client.chat.completions.create(
        model="gpt-5.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return completion.choices[0].message.content.strip()

# ==========================
#   LECTURA / ESCRITURA JSONL + PRINTS
# ==========================
input_path = "RAG.jsonl"
output_path = "responses-10.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for idx, line in enumerate(fin):
        task = json.loads(line)

        gen = generate_response(task)
        target = get_target_answer(task)

        # formato exacto requerido
        task["predictions"] = [{"text": gen}]

        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

        # prints como antes
        print(f"=== TURNO {task.get('turn', idx)} ===")
        print(f"PREGUNTA: {get_current_question(task)}")
        print(f"RESPUESTA GENERADA: {gen}")
        print(f"RESPUESTA OBJETIVO: {target}")
        print("-" * 80)

print("✅ responses-10.jsonl generado correctamente.")


Se truncaron las últimas líneas 5000 del resultado de transmisión.
The main factor is a long history of on‑time payments; there is no quick fix beyond that.
RESPUESTA OBJETIVO:  Building credit generally means building up your FICO score. Your FICO score is impacted my many factors, one of which is your utilization ratio of your installment loans like student loans. This is the ratio of the current balance to your original balance. To improve your score (slightly) you would want a lower ratio.  You can use some services  that specifically help you boost your credit card score at boostcredit101.com.  They add you as an authorized user to a credit card with a high limit, low balance, and perfect payment history.  You can also request a line of credit increase yourself without waiting for the bank to do so. Use your credit card regularly and also repay  your bills on time. This will help increase your credit score. 
-------------------------------------------------------------------------

In [ ]:
# pip install rouge_score

## Metrica RG_agg y RB_llm

In [ ]:
# ============================================
#  QUICK GENERATION EVALUATION (COLAB)
#  Metrics:
#   - RB_agg (numeric, Rouge-L + EM)
#   - RB_llm (OpenAI judge)
# ============================================

import json
import numpy as np
from rouge_score import rouge_scorer
from openai import OpenAI
from tqdm import tqdm

# ==========================
# CONFIG
# ==========================
INPUT_JSONL = "responses-10.jsonl"   # <- tu archivo con predictions
OPENAI_MODEL = "gpt-5.1"             # juez
OPENAI_API_KEY = ""

client = OpenAI(api_key=OPENAI_API_KEY)

# ==========================
# HELPERS
# ==========================
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def harmonic_mean(vals):
    vals = [v for v in vals if v > 0]
    if len(vals) == 0:
        return 0.0
    return len(vals) / sum(1.0 / v for v in vals)

def get_reference(task):
    t = task.get("targets", [])
    if isinstance(t, list) and len(t) > 0:
        return t[0].get("text", "")
    return ""

def get_prediction(task):
    p = task.get("predictions", [])
    if isinstance(p, list) and len(p) > 0:
        return p[0].get("text", "")
    return ""

# ==========================
# RB_AGG (NUMERIC)
# ==========================
def compute_rb_agg(pred, ref):
    if not pred or not ref:
        return 0.0

    rougeL = scorer.score(ref, pred)["rougeL"].fmeasure
    exact_match = float(pred.strip().lower() == ref.strip().lower())

    # Paper-style aggregation (simple + fast)
    return harmonic_mean([rougeL, max(exact_match, 1e-6)])

# ==========================
# RB_LLM (OPENAI JUDGE)
# ==========================
JUDGE_PROMPT = """
You are an expert evaluator for Retrieval-Augmented Generation.

Compare the MODEL RESPONSE to the REFERENCE ANSWER.

Evaluate based on:
- Faithfulness
- Appropriateness
- Completeness

Give a single numeric score from 0 to 10.
ONLY output the number.

REFERENCE ANSWER:
{reference}

MODEL RESPONSE:
{prediction}
"""

def openai_judge(pred, ref):
    if not pred:
        return 0.0

    prompt = JUDGE_PROMPT.format(reference=ref, prediction=pred)

    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )

    text = resp.choices[0].message.content.strip()
    try:
        score = float(text)
        return np.clip(score / 10.0, 0.0, 1.0)
    except:
        return 0.0

# ==========================
# MAIN LOOP
# ==========================
rb_agg_scores = []
rb_llm_scores = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    tasks = [json.loads(line) for line in f]

print(f"Evaluating {len(tasks)} examples...\n")

for task in tqdm(tasks):
    ref = get_reference(task)
    pred = get_prediction(task)

    rb_agg = compute_rb_agg(pred, ref)
    rb_llm = openai_judge(pred, ref)

    rb_agg_scores.append(rb_agg)
    rb_llm_scores.append(rb_llm)

    # guarda métricas por ejemplo (opcional)
    task.setdefault("metrics", {})
    task["metrics"]["RB_agg"] = rb_agg
    task["metrics"]["RB_llm"] = rb_llm

# ==========================
# RESULTS
# ==========================
print("\n====== RESULTS ======")
print(f"RB_agg (mean): {np.mean(rb_agg_scores):.4f}")
print(f"RB_llm (mean): {np.mean(rb_llm_scores):.4f}")

# ==========================
# SAVE EVAL FILE
# ==========================
with open("eval_results.jsonl", "w", encoding="utf-8") as f:
    for t in tasks:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print("\n✅ Saved: eval_results.jsonl")


Evaluating 842 examples...



100%|██████████| 842/842 [10:17<00:00,  1.36it/s]


====== RESULTS ======
RB_agg (mean): 0.0036
RB_llm (mean): 0.6771

✅ Saved: eval_results.jsonl


In [ ]:
# 30 minutos 3.5 dolares
# 15 minutos medio dolar

In [ ]:
# QUe le mejoraria, revisar por cada uno, todo de corrido.

In [ ]:
# Con reference y la de cohere, primero pruebas pequeñas

In [ ]:
!pip install -q cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 28.4 MB/s eta 0:00:00


In [ ]:
# ============================================
#  COHERE LLM JUDGE (RB_llm) - COLAB
#  Model: command-a-03-2025
# ============================================

import json
import numpy as np
import cohere
from tqdm import tqdm

# ==========================
INPUT_JSONL = "responses-10 (1).jsonl"     # archivo con predictions
OUTPUT_JSONL = "eval_results_cohere.jsonl"

COHERE_API_KEY = ""
COHERE_MODEL = "command-a-03-2025"

co = cohere.ClientV2(api_key=COHERE_API_KEY)

# ==========================
def get_reference(task):
    t = task.get("targets", [])
    if isinstance(t, list) and len(t) > 0:
        return t[0].get("text", "")
    return ""

def get_prediction(task):
    p = task.get("predictions", [])
    if isinstance(p, list) and len(p) > 0:
        return p[0].get("text", "")
    return ""

# ==========================
# COHERE JUDGE PROMPT (RADBENCH-STYLE)
# ==========================
JUDGE_PROMPT = """
You are an expert evaluator for Retrieval-Augmented Generation.

Compare the MODEL RESPONSE to the REFERENCE ANSWER.

Evaluate the response based on:
- Faithfulness
- Appropriateness
- Completeness

Give a single numeric score from 0 to 10.
ONLY output the number.

REFERENCE ANSWER:
{reference}

MODEL RESPONSE:
{prediction}
"""

def cohere_judge(pred, ref):
    if not pred:
        return 0.0

    prompt = JUDGE_PROMPT.format(
        reference=ref,
        prediction=pred
    )

    resp = co.chat(
        model=COHERE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.0,
    )

    # Cohere V2 devuelve bloques
    text_out = ""
    for block in resp.message.content:
        if block.type == "text":
            text_out += block.text

    text_out = text_out.strip()

    try:
        score = float(text_out)
        return np.clip(score / 10.0, 0.0, 1.0)
    except:
        return 0.0

# ==========================
# MAIN LOOP
# ==========================
rb_llm_cohere_scores = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    tasks = [json.loads(line) for line in f]

print(f"Evaluating {len(tasks)} examples with Cohere judge...\n")

for task in tqdm(tasks):
    ref = get_reference(task)
    pred = get_prediction(task)

    rb_llm = cohere_judge(pred, ref)
    rb_llm_cohere_scores.append(rb_llm)

    task.setdefault("metrics", {})
    task["metrics"]["RB_llm_cohere"] = rb_llm

# ==========================
# RESULTS
# ==========================
print("\n====== COHERE JUDGE RESULTS ======")
print(f"RB_llm_cohere (mean): {np.mean(rb_llm_cohere_scores):.4f}")

# ==========================
# SAVE
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for t in tasks:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print(f"\n✅ Saved: {OUTPUT_JSONL}")


Evaluating 842 examples with Cohere judge...



100%|██████████| 842/842 [03:11<00:00,  4.40it/s]



====== COHERE JUDGE RESULTS ======
RB_llm_cohere (mean): 0.7207

✅ Saved: eval_results_cohere.jsonl


In [ ]:
# !pip install -q ragas langchain-openai datasets tiktoken

## Métrica RAGAS

In [ ]:
# 70 centavos de dolar 15 minutos

In [ ]:
# ============================================
#  RAGAS LLM JUDGE → RLF (Faithfulness)
#  Uses OpenAI as LLM
#  Reads JSONL with predictions
# ============================================

import json
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI

# ==========================
# CONFIG
# ==========================
INPUT_JSONL = "responses-10 (1).jsonl"
OUTPUT_JSONL = "eval_results_ragas.jsonl"

OPENAI_API_KEY = ""
OPENAI_JUDGE_MODEL = "gpt-4o-mini"

# ==========================
# HELPERS
# ==========================
def extract_prediction(task):
    p = task.get("predictions", [])
    if isinstance(p, list) and len(p) > 0:
        return p[0].get("text", "")
    return ""

def extract_context_texts(task):
    """
    RAGAS espera: List[str] por ejemplo
    """
    ctxs = task.get("contexts", [])
    texts = []
    for c in ctxs:
        t = c.get("text", "")
        if t:
            texts.append(t)
    return texts

def extract_question(task):
    """
    RAGAS necesita una 'question', aunque
    no la usa directamente para faithfulness.
    Tomamos la última pregunta del input.
    """
    msgs = task.get("input", [])
    for m in reversed(msgs):
        if m.get("speaker") == "user":
            return m.get("text", "")
    return ""

# ==========================
# LOAD DATA
# ==========================
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    tasks = [json.loads(line) for line in f]

questions = []
answers = []
contexts = []

for t in tasks:
    questions.append(extract_question(t))
    answers.append(extract_prediction(t))
    contexts.append(extract_context_texts(t))

dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
})

# ==========================
# RAGAS EVALUATION
# ==========================
llm = ChatOpenAI(
    model=OPENAI_JUDGE_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0.0,
)

print(f"Running RAGAS Faithfulness on {len(dataset)} examples...\n")

result = evaluate(
    dataset,
    metrics=[faithfulness],
    llm=llm,
    run_config=RunConfig(timeout=120),
)

df_scores = result.to_pandas()

# ==========================
# ATTACH SCORES BACK
# ==========================
rlf_scores = df_scores["faithfulness"].values

for task, score in zip(tasks, rlf_scores):
    task.setdefault("metrics", {})
    task["metrics"]["RLF"] = float(score)

# ==========================
# RESULTS
# ==========================
print("\n====== RAGAS (RLF) RESULTS ======")
print(f"RLF (mean): {np.mean(rlf_scores):.4f}")

# ==========================
# SAVE
# ==========================
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for t in tasks:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print(f"\n✅ Saved: {OUTPUT_JSONL}")

Running RAGAS Faithfulness on 842 examples...



Evaluating:   0%|          | 0/842 [00:00<?, ?it/s]


====== RAGAS (RLF) RESULTS ======
RLF (mean): 0.7536

✅ Saved: eval_results_ragas.jsonl
